In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt

from ALLCools.mcds import MCDS
from ALLCools.clustering import tsne, significant_pc_test, log_scale
from ALLCools.plot import *

In [ ]:
# change this to the path to your metadata
metadata_path = 'CellMetadata.PassQC.Quintile.correctCellNames.csv.gz'

# Basic filtering parameters. 
# These are suggesting values, cutoff maybe different for different tissue and sequencing depths.
# To determine each cutoff more appropriately, one need to plot the distribution of each metric.
mapping_rate_cutoff = 0.5
mapping_rate_col_name = 'R1MappingRate'  # Name may change
final_reads_cutoff = 500000
final_reads_col_name = 'FinalmCReads'  # Name may change
mccc_cutoff = 0.03
mccc_col_name = 'mCCCFrac'  # Name may change
mch_cutoff = 0.2
mch_col_name = 'mCHFrac'  # Name may change
mcg_cutoff = 0.05
mcg_col_name = 'mCGFrac'  # Name may change

# change this to the paths to your MCDS files, 
# ALLCools.MCDS can handle multiple MCDS files automatically
mcds_path = './GSC_epitherapy_with5k.mcds/'

# Dimension name used to do clustering
# This corresponding to AnnData .obs and .var
obs_dim = 'cell'  # observation
var_dim = 'chrom100k'  # feature

# feature cov cutoffs
min_cov = 500
max_cov = 3000

# Regions to remove during the clustering analysis
# change this to the path to ENCODE blacklist.
# The ENCODE blacklist can be downloaded from https://github.com/Boyle-Lab/Blacklist/
black_list_path = '/scratch/devtools/mmoore/genomes/snm3c/hg38/hg38-blacklist.v2.bed.gz'
black_list_fraction = 0.2
exclude_chromosome = ['chrM', 'chrY']

# load to memory or not
load = True

# HVF
mch_pattern = 'CHN'
mcg_pattern = 'CGN'
n_top_feature = 20000

# PC cutoff
pc_cutoff = 0.1

# KNN
knn = -1  # -1 means auto determine

# Leiden
resolution = 2

In [ ]:
metadata = pd.read_csv(metadata_path, index_col=0)
print(f'Metadata of {metadata.shape[0]} cells')
metadata.head()

In [ ]:
judge = (metadata[mapping_rate_col_name] > mapping_rate_cutoff) & \
        (metadata[final_reads_col_name] > final_reads_cutoff) & \
        (metadata[mccc_col_name] < mccc_cutoff) & \
        (metadata[mch_col_name] < mch_cutoff) & \
        (metadata[mcg_col_name] > mcg_cutoff)

metadata = metadata[judge].copy()
print(f'{metadata.shape[0]} cells passed filtering')

In [ ]:
mcds = MCDS.open(
    mcds_path, 
    obs_dim='cell', 
    var_dim='chrom100k',
    use_obs=metadata.index  # MCDS contains all cells, this will select cells that passed filtering 
)
total_feature = mcds.get_index(var_dim).size
mcds

In [ ]:
# you can add the cell metadata into MCDS
mcds.add_cell_metadata(metadata)

In [ ]:
mcds.add_feature_cov_mean(var_dim=var_dim)

In [ ]:
# filter by coverage - based on the distribution above
mcds = mcds.filter_feature_by_cov_mean(
    min_cov=min_cov,  # minimum coverage
    max_cov=max_cov  # maximum coverage
)

# remove blacklist regions
mcds = mcds.remove_black_list_region(
    black_list_path=black_list_path,
    f=black_list_fraction  # Features having overlap > f with any black list region will be removed.
)

# remove chromosomes
mcds = mcds.remove_chromosome(exclude_chromosome)

In [ ]:
mcds.add_mc_frac(
normalize_per_cell=False,  # after calculating mC frac, per cell normalize the matrix
    #clip_norm_value=10  # clip outlier values above 10 to 10
)

# load only the mC fraction matrix into memory so following steps is faster
# Only load into memory when your memory size is enough to handle your dataset
if load and (mcds.get_index(obs_dim).size < 20000):
    mcds[f'{var_dim}_da_frac'].load()

In [ ]:
mch_hvf = mcds.calculate_hvf_svr(var_dim=var_dim,
                                 mc_type=mch_pattern,
                                 n_top_feature=n_top_feature,
                                 plot=True)

In [ ]:
mcg_hvf = mcds.calculate_hvf_svr(var_dim=var_dim,
                                 mc_type=mcg_pattern,
                                 n_top_feature=n_top_feature,
                                 plot=True)

In [ ]:
mch_adata = mcds.get_adata(mc_type=mch_pattern,
                           var_dim=var_dim,
                           select_hvf=True)
mch_adata

In [ ]:
mcg_adata = mcds.get_adata(mc_type=mcg_pattern,
                           var_dim=var_dim,
                           select_hvf=True)
mcg_adata

In [ ]:
log_scale(mch_adata)

In [ ]:
log_scale(mcg_adata)

In [ ]:
sc.tl.pca(mch_adata)
ch_n_components = significant_pc_test(mch_adata)
fig, axes = plot_decomp_scatters(mch_adata,
                                 n_components=ch_n_components,
                                 hue=mch_col_name,
                                 hue_quantile=(0.25, 0.75),
                                 nrows=3,
                                 ncols=5)

In [ ]:
sc.tl.pca(mcg_adata)
cg_n_components = significant_pc_test(mcg_adata)
fig, axes = plot_decomp_scatters(mcg_adata,
                                 n_components=cg_n_components,
                                 hue=mcg_col_name,
                                 hue_quantile=(0.25, 0.75),
                                 nrows=3,
                                 ncols=5)

In [ ]:
ch_pcs = mch_adata.obsm['X_pca'][:, :ch_n_components]
cg_pcs = mcg_adata.obsm['X_pca'][:, :cg_n_components]

# scale the PCs so CH and CG PCs has the same total var
cg_pcs = cg_pcs / cg_pcs.std()
ch_pcs = ch_pcs / ch_pcs.std()

# total_pcs
#total_pcs = np.hstack([ch_pcs, cg_pcs])
total_pcs=cg_pcs

# make a copy of adata, add new pcs
# this is suboptimal, will change this when adata can combine layer and X in the future
adata = mch_adata.copy()
adata.obsm['X_pca'] = total_pcs
del adata.uns['pca']
del adata.varm['PCs']

In [ ]:
if knn == -1:
    knn = max(15, int(np.log2(adata.shape[0])*2))
sc.pp.neighbors(adata, n_neighbors=knn)

In [ ]:
sc.tl.leiden(adata, resolution=resolution)

In [ ]:
tsne(adata,
     obsm='X_pca',
     metric='euclidean',
     exaggeration=-1,  # auto determined
     perplexity=30,
     n_jobs=-1)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
_ = categorical_scatter(data=adata,
                        ax=ax,
                        coord_base='tsne',
                        hue='leiden',
                        text_anno='leiden',
                        show_legend=True)

In [ ]:
sc.tl.umap(adata)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
_ = categorical_scatter(data=adata,
                        ax=ax,
                        coord_base='umap',
                        hue='leiden',
                        text_anno='leiden',
                        show_legend=True)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
_ = categorical_scatter(data=adata,
                        ax=ax,
                        coord_base='umap',
                        hue='Cell_line',
                        #text_anno='leiden',
                        show_legend=True)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
_ = categorical_scatter(data=adata,
                        ax=ax,
                        coord_base='umap',
                        hue='Treatment',
                        #text_anno='leiden',
                        show_legend=True)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
_ = continuous_scatter(data=adata,
                        ax=ax,
                        coord_base='umap',
                        hue='mCGFrac')
                        #text_anno='leiden',
                        #show_legend=True)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
_ = categorical_scatter(data=adata,
                        ax=ax,
                        coord_base='umap',
                        hue='qcluster',
                        text_anno='leiden',
                        show_legend=True)

In [ ]:
# in order to reduce the page size, I downsample the data here, you don't need to do this
interactive_scatter(data=adata,
                    hue='leiden',
                    coord_base='umap',
                    max_points=3000)

In [ ]:
adata.write_h5ad('GSC.chrom100k-clustering.h5ad')
adata

In [ ]:
adata.obs.to_csv('GSC.ClusteringResults100k.csv.gz')
adata.obs.head()

In [ ]:
print(
    f'{mcds.get_index(var_dim).size} ({mcds.get_index(var_dim).size * 100 / total_feature:.1f}%) '
    f'{var_dim} remained after all the basic filter.')

In [ ]:
with open('FeatureList.BasicFilter.txt', 'w') as f:
    for var in mcds.get_index(var_dim).astype(str):
        f.write(var + '\n')

In [ ]:
mcg_adata.X

In [ ]:
df = pd.DataFrame(
    mcg_adata.X,
    index=mcg_adata.obs_names,
    columns=mcg_adata.var_names
)

In [ ]:
df

In [ ]:
from sklearn.metrics.pairwise import pairwise_distances
dist = pairwise_distances(df, metric="cosine")
dist

In [ ]:
import seaborn as sns
g = sns.clustermap(
    dist,
    cmap="viridis",
    row_cluster=True,
    col_cluster=True,
    #row_colors=row_colors,
    #col_colors=row_colors,
    figsize=(10,10)
)

In [ ]:
dist_df = pd.DataFrame(dist, index=mcg_adata.obs_names.values, columns=mcg_adata.obs_names.values)
dist_df.to_csv("cell_cell_euclidean.tsv", sep="\t")

In [ ]:
da = mcds['chrom100k_da_frac'].values

In [ ]:
data = da[:, :, 2]
data

In [ ]:
cells = mcds['cell'].values.astype(str)
cells

In [ ]:
df = pd.DataFrame(data, index=cells, columns=mcds['chrom100k'].values)
df

In [ ]:
from sklearn.metrics.pairwise import pairwise_distances
dist = pairwise_distances(df, metric="cosine")
dist

In [ ]:
import seaborn as sns
g = sns.clustermap(
    dist,
    cmap="viridis",
    row_cluster=True,
    col_cluster=True,
    #row_colors=row_colors,
    #col_colors=row_colors,
    figsize=(10,10)
)

In [ ]:
dist_df = pd.DataFrame(dist, index=cells, columns=cells)
dist_df.to_csv("cell_cell_euclidean.tsv", sep="\t")